# Exploratory Data Analysis for Crop Production Prediction

This notebook performs exploratory data analysis on the crop production dataset for India. We'll load the data, clean it, visualize key insights, and prepare it for machine learning modeling.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Load and Explore Dataset

Load the crop production dataset and examine its structure, missing values, and basic statistics.

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/crop_production.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Basic statistics
print("\nBasic Statistics:")
df.describe()

## Data Preprocessing

Handle missing values, encode categorical variables, and prepare the data for modeling.

In [ ]:
# Handle missing values (if any)
df = df.dropna()  # For simplicity, drop rows with missing values

# Encode categorical variables
le = LabelEncoder()
categorical_cols = ['State_Name', 'District_Name', 'Season', 'Crop']
for col in categorical_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])

# Select features and target
features = ['Crop_Year', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Temperature'] + [col + '_encoded' for col in categorical_cols]
target = 'Production'

X = df[features]
y = df[target]

# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['Crop_Year', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Temperature']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## Exploratory Data Analysis

Visualize the data to understand relationships and patterns.

In [ ]:
# Crop production trends over years
plt.figure(figsize=(12, 6))
yearly_production = df.groupby('Crop_Year')['Production'].sum()
plt.plot(yearly_production.index, yearly_production.values, marker='o')
plt.title('Crop Production Trends Over Years')
plt.xlabel('Year')
plt.ylabel('Total Production')
plt.grid(True)
plt.show()

In [ ]:
# Rainfall vs Production scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(df['Annual_Rainfall'], df['Production'], alpha=0.5)
plt.title('Rainfall vs Production')
plt.xlabel('Annual Rainfall (mm)')
plt.ylabel('Production')
plt.grid(True)
plt.show()

In [ ]:
# State-wise production
plt.figure(figsize=(14, 8))
state_production = df.groupby('State_Name')['Production'].sum().sort_values(ascending=False).head(10)
state_production.plot(kind='bar')
plt.title('Top 10 States by Total Production')
plt.xlabel('State')
plt.ylabel('Total Production')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df[['Crop_Year', 'Area', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Temperature', 'Production']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## Feature Engineering

Create additional features if needed (in this case, the basic features are sufficient).

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

## Train Machine Learning Models

Train multiple regression models on the training data.

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
}

# Train models
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully")

## Evaluate and Compare Models

Evaluate the trained models using various metrics.

In [ ]:
# Evaluate models
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

# Display results
results_df = pd.DataFrame(results).T
print("Model Evaluation Results:")
results_df

In [ ]:
# Visualize model comparison
results_df.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Save the Best Model

Save the best performing model for use in the web application.

In [ ]:
# Select best model (highest R² score)
best_model_name = results_df['R²'].idxmax()
best_model = trained_models[best_model_name]

# Save the model
joblib.dump(best_model, '../models/crop_model.pkl')
print(f"Best model ({best_model_name}) saved as crop_model.pkl")

# Also save the scaler and label encoders for preprocessing in the app
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
print("Scaler and label encoder saved")

## Summary

This notebook has successfully:
1. Loaded and explored the crop production dataset
2. Performed data preprocessing including encoding and scaling
3. Conducted exploratory data analysis with visualizations
4. Trained multiple machine learning models
5. Evaluated and compared model performance
6. Saved the best model for deployment

The best model can now be used in the Streamlit web application for crop production prediction.